#Le problème de Monty Hall

In [21]:
import numpy as np
import matplotlib.pyplot as plt

##Introduction
Il s'agit d'une émission de télévision où l'on peut gager une voiture en jouant à un jeu. Le principe est : il y a 3 portes fernées, l'une d'elles cache une voiture, les deux autres des chèvres.
Le jeu est en 2 étapes :
1. L'animateur laisse choisir une des trois portes, mais nous ne l'ouvrons pas encore.
2. L'animateur (qui sait où se trouve la voiture) choisit l'une des deux portes restantes et l'ouvre, révélant une chèvre.

**Souhaitez-vous changer de porte ou choisir l'autre porte ?**

##Tester le jeu

Il s'agira de créer une fonction monty_hall intéractive. Le joueur va pouvoir jouer autant de fois qu'il le souhaite. Il choisira sa porte, changer ou rester. A la fin sera calculer le nombre de victoire et d'échec.

In [28]:
def monty_hall_interactif():
    #Pour commencer on initialise le tout
    victoires = 0
    defaites = 0
    continuer = 'o'

    while continuer == 'o':
        """
        La première étape consiste à placer la voiture derrière une porte, pour cela :
        - on initialise en donnant comme valeur 0 à toutes les portes, cela correspond à des chèvres
        - on choisit au hasard le numéro de la porte qui aura la voiture
        - on place la voiture derrière une porte en attribuant la valeur 1
        """
        portes = np.array([0, 0, 0])
        index_gagnant = np.random.randint(0, 3)
        portes[index_gagnant] = 1

        choix = int(input("Choisissez une porte (0, 1 ou 2) : "))
        portes_disponibles = [i for i in range(3) if i not in (index_gagnant, choix)] #permet de savoir quelle porte peut être ouverte
        porte_ouverte = np.random.choice(portes_disponibles) #Choisit la porte à ouvrir
        print(f"L'animateur ouvre la porte {porte_ouverte} — c'est une chèvre !")

        decision = input("Voulez-vous changer de porte ? (o/n) : ")
        if decision == 'o':
            choix = [i for i in range(3) if i not in (choix, porte_ouverte)][0]

        if portes[choix] == 1:
            print(f"Porte {choix} — Gagné !")
            victoires += 1
        else:
            print(f"Porte {choix} — Perdu ! La voiture était derrière la porte {index_gagnant}.")
            defaites += 1

        continuer = input("Rejouer ? (o/n) : ")

    print(f"\nBilan — Victoires : {victoires} | Défaites : {defaites}")

In [27]:
monty_hall_interactif()

Choisissez une porte (0, 1 ou 2) : 1
L'animateur ouvre la porte 0 — c'est une chèvre !
Voulez-vous changer de porte ? (o/n) : o
Porte 2 — Gagné !
Rejouer ? (o/n) : o
Choisissez une porte (0, 1 ou 2) : 0
L'animateur ouvre la porte 2 — c'est une chèvre !
Voulez-vous changer de porte ? (o/n) : n
Porte 0 — Gagné !
Rejouer ? (o/n) : o
Choisissez une porte (0, 1 ou 2) : 2
L'animateur ouvre la porte 0 — c'est une chèvre !
Voulez-vous changer de porte ? (o/n) : n
Porte 2 — Perdu ! La voiture était derrière la porte 1.
Rejouer ? (o/n) : n

Bilan — Victoires : 2 | Défaites : 1


##Simulation du jeu pendant plusieurs parties

Cette fonction va permettre de simuler le jeu pendant de nombreuses intérations afin de déterminer si le taux de réussite varie d'une stratégie à l'autre.
Il sera alors possible de choisir le nombre de parties et la stratégie : rester ou changer.

In [48]:
def simuler_monty_hall(n_parties, changer):
    gains = 0
    for _ in range(n_parties):
        # Placement aléatoire de la voiture
        voiture = np.random.randint(0, 3)
        # Choix initial du joueur
        choix = np.random.randint(0, 3)

        if changer:
            # L'animateur ouvre une porte avec une chèvre
            # Le joueur change — il gagne si son choix initial était faux
            gains += (choix != voiture)
        else:
            # Le joueur reste — il gagne si son choix initial était juste
            gains += (choix == voiture)

    return gains / n_parties



In [49]:
print(f"P(gagner en changeant)   = {simuler_monty_hall(n_parties=10000, changer=True):.3f}")
print(f"P(gagner en restant)     = {simuler_monty_hall(n_parties=10000, changer=False):.3f}")

P(gagner en changeant)   = 0.660
P(gagner en restant)     = 0.338


La meilleur stratégie consiste à **changer de porte**

Au départ, **la probabilité de choisir la bonne porte est de 1/3**. PAr conséquent, la probabilité que la voiture soit derrière l'une des deux autres portes est donc de 2/3.


Quand l'animateur ouvre une porte avec une chèvre, il ne change pas cette répartition.  Il concentre simplement les 2/3 de probabilité sur la seule porte restante.


**Changer de porte revient à parier sur les 2/3**. Rester revient à parier sur le 1/3 initial.

La simulation sur 10 000 parties confirme : P(gagner en changeant) ≈ 0.667, P(gagner en restant) ≈ 0.333.

## Généralisation — n portes, k ouvertes
Le problème classique de Monty Hall peut être généralisé à n portes,
dont l'hôte en ouvre k (toujours des chèvres, jamais ta porte ni la voiture).

La formule théorique devient : P(gagner en changeant) = (n-1) / (n × (n-k-1))

Avec n=3 et k=1 on retrouve le résultat classique : 2/3.
Plus k est grand, plus changer devient avantageux —
l'hôte a révélé beaucoup d'information.

In [47]:
def simuler_monty_hall_general(n_parties, n_portes, k_ouvertes, changer):
    gains = 0
    for _ in range(n_parties):
        voiture = np.random.randint(0, n_portes)
        choix = np.random.randint(0, n_portes)

        if changer:
            # Portes restantes après que l'hôte en ouvre k
            portes_restantes = n_portes - k_ouvertes - 1
            if choix != voiture:
                # Le joueur change et tombe sur la bonne porte
                # avec probabilité 1/(n-k-1)
                gains += (np.random.randint(0, portes_restantes) == 0)
        else:
            gains += (choix == voiture)

    return gains / n_parties

# Cas classique : 3 portes, 1 ouverte
print(f"3 portes, changer : {simuler_monty_hall_general(10000, 3, 1, True):.3f}")
print(f"3 portes, rester  : {simuler_monty_hall_general(10000, 3, 1, False):.3f}")

# Cas généralisé : 5 portes, 2 ouvertes
print(f"5 portes, changer : {simuler_monty_hall_general(10000, 5, 2, True):.3f}")
print(f"5 portes, rester  : {simuler_monty_hall_general(10000, 5, 2, False):.3f}")

3 portes, changer : 0.665
3 portes, rester  : 0.340
5 portes, changer : 0.406
5 portes, rester  : 0.197


La règle est universelle : **il est toujours préférable de changer**,
sauf si k=0 (l'hôte n'ouvre aucune porte — aucune information nouvelle).